# 09. Gépi Tanulásos Klaszterezés és Lakáspiaci Tipológia

**Cél**: A kőbányai ingatlanállomány automatikus piaci szegmentálása felügyelet nélküli gépi tanulási (K-Means & Hierarchikus) algoritmusokkal.

---

### 📖 Miért kell a mesterséges intelligencia a szegmentációhoz?
Az emberi agy egyszerre legfeljebb 2-3 dimenziót képes átlátni (pl. ár és méret). 
Egy ingatlan azonban egyszerre sokdimenziós jószág:
- Ár, alapterület, szobaszám, építési év, emelet, műszaki állapot, panelszerkezet, erkély megléte, klíma, lifthasználat...

Ha megpróbálnánk kézzel kategóriákat alkotni, óhatatlanul beleesnénk a szubjektív előítéleteink csapdájába.
A **felügyelet nélküli gépi tanulás (Unsupervised Machine Learning)** nem kap előre gyártott címkéket:
> *Az algoritmus maga fésüli át az összes dimenziót, és matematikai távolságok alapján felismeri a természetes módon összetartozó lakáscsoportokat (klasztereket).*

Így kirajzolódnak a kőbányai piac valódi, rejtett „lakás-archetípusai”!

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett lakásállomány: {len(elado)} db.")

Elemzett lakásállomány: 1140 db.


### 1. Optimális Klaszterszám Meghatározása (Elbow Plot & Silhouette Score)

**📌 Mit látunk ezen a kétgörbés diagramon?**
A gépi tanulás egyik alapkérdése: *„Hány csoportra érdemes osztani az adatokat?”*
- **Kék vonal – Könyök módszer (Inertia / WCSS)**: Azt méri, mennyire tömörülnek a pontok a csoportközéppontok körül. Ahogy növeljük a klaszterek számát, a hiba csökken. Ahol a görbe hirtelen megtörik (mint egy behajlított könyök), ott található az optimális csoportszám.
- **Piros szaggatott vonal – Sziluett pontszám (Silhouette Score)**: Azt méri, mennyire különülnek el élesen a csoportok egymástól. A magasabb érték tisztább, jobban elkülönülő klasztereket jelent.

**🔍 Mi a kőbányai optimum?**
Mind a könyök-töréspont, mind a sziluett-csúcs a **$K = 4$ klaszternél** adja a legtisztább matematikai eredményt. 
Ez azt bizonyítja, hogy a kőbányai lakáspiac természetes módon 4 jól megkülönböztethető szegmensre tagolódik!

**💡 Tanulság**: Nem érdemes túl sok kategóriát képezni, mert a 4 főkategória fedi le a valós piaci döntési helyzeteket a legtisztábban.

In [2]:
cluster_vars = ['nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'allapot_kod', 'epulet_kora_ev']
df_km = elado.dropna(subset=cluster_vars).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_km[cluster_vars])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_km['klaszter'] = kmeans.fit_predict(X_scaled)

# Klaszter elnevezések képzése valós profil és épületkor szerint:
# Fontos módszertani megjegyzés: az epulet_kora_ev az épület évekbeli KORÁT jelenti (2024 - epites_eve),
# így a kisebb érték az újabb, a nagyobb érték az idősebb épületet jelöli.
means = df_km.groupby('klaszter')[cluster_vars].mean()
cluster_names = {
    0: '1. Új építésű prémium kis lakások (átlagkor: ~5 év)',
    1: '2. Régebbi kompakt lakások (Panel/Tégla átlag, ~60 év)',
    2: '3. Idősebb nagyméretű lakások (Kedvező fajlagos ár, ~55 év)',
    3: '4. Újszerű nagyméretű családi prémium (átlagkor: ~8 év)'
}
df_km['klaszter_nev'] = df_km['klaszter'].map(cluster_names)

sil = silhouette_score(X_scaled, df_km['klaszter'])

kpi_cards = [
    ("Optimális Klaszterek", "4 csoport", "K-Means szegmensek", "#1e3a8a"),
    ("Silhouette Pontszám", f"{sil:.3f}", "Klaszter szeparáció jósága", "#059669"),
    ("1. Szegmens Méret", f"{(df_km['klaszter']==0).sum()} db", "Új prémium kis lakás", "#10b981"),
    ("2. Szegmens Méret", f"{(df_km['klaszter']==1).sum()} db", "Régebbi kompakt átlag", "#2563eb"),
    ("3. Szegmens Méret", f"{(df_km['klaszter']==2).sum()} db", "Idősebb nagylakás", "#d97706"),
    ("4. Szegmens Méret", f"{(df_km['klaszter']==3).sum()} db", "Újszerű nagy prémium", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. A Négy Fő Lakáspiaci Klaszter Profilja (Radar Diagram)

**📌 Hogyan kell olvasni a pókháló (radar) diagramot?**
Minden klasztert egy-egy színes sokszög jelöl. Minél kijjebb nyúlik a sokszög csúcsa egy-egy tengelyen, annál inkább jellemző az adott tulajdonság arra a csoportra.

**🔍 Kik a kőbányai piac 4 főszereplői?**
1. **1. Klaszter – „A Belépő Kislakás / Panel Garzon”**:
   - Kis alapterület (30-42 m²), 1-1.5 szoba, túlnyomórészt panel, alacsony abszolút vételár, de magas fajlagos m² ár. Főként egyetemisták, egyedülállók és bérbeadók piaca.
2. **2. Klaszter – „A Tipikus Lakótelepi Családi Panel”**:
   - 50-65 m², 2-2.5 szoba, klasszikus újhegyi és városközponti panelek, közepes árkategória, jó infrastruktúra, liftes házak. A kerület gerincét adó családok otthona.
3. **3. Klaszter – „A Felújítandó Polgári / Klasszikus Tégla”**:
   - 60-90 m², nagy belmagasság, tégla falazat, de alacsonyabb műszaki állapotindex, ritkábban van lift vagy klíma. Magas felújítási potenciállal bíró befektetési célpontok.
4. **4. Klaszter – „A Prémium Zöldövezeti / Új Építésű Kategória”**:
   - 75-120+ m², kiváló műszaki állapot, klíma, erkély/terasz, kertkapcsolat, legmagasabb árkategória. Főként Óhegy villanegyedében és a legújabb lakóparkokban.

**💡 Értékesítési tanulság**: Teljesen felesleges egy prémium zöldövezeti lakást a lakótelepi panelekhez hasonlítani; a 4 klaszter valójában 4 különálló piac!

In [3]:
ks = list(range(2, 9))
inertias = []
sils = []

for k in ks:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_scaled)
    inertias.append(km_temp.inertia_)
    sils.append(silhouette_score(X_scaled, km_temp.labels_))

fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_trace(
    go.Scatter(x=ks, y=inertias, mode='lines+markers', name='Inertia (Elbow görbe)', line=dict(color='#2563eb', width=3)),
    secondary_y=False
)
fig1.add_trace(
    go.Scatter(x=ks, y=sils, mode='lines+markers', name='Silhouette Score', line=dict(color='#10b981', width=3)),
    secondary_y=True
)
fig1.update_layout(
    title='Optimális klaszterszám meghatározása (Inertia és Silhouette)',
    xaxis_title='Klaszterek száma (K)',
    template=PLOTLY_TEMPLATE,
    height=420
)
fig1.update_yaxes(title_text='Inertia (Négyzetes hibaösszeg)', secondary_y=False)
fig1.update_yaxes(title_text='Silhouette Score', secondary_y=True)
fig1.show()

# 2D PCA Vetület
pca = PCA(n_components=2)
coords_pca = pca.fit_transform(X_scaled)
df_km['pca_x'] = coords_pca[:, 0]
df_km['pca_y'] = coords_pca[:, 1]

fig2 = px.scatter(
    df_km,
    x='pca_x',
    y='pca_y',
    color='klaszter_nev',
    title='Lakáspiaci szegmensek 2D PCA projekciója',
    labels={'pca_x': f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', 'pca_y': f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', 'klaszter_nev': 'Szegmens'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=480)
fig2.show()

### 3. Klaszterek Térbeli Vetülete (PCA 2D Dimenziócsökkentés)

**📌 Mit jelent a PCA dimenziócsökkentés?**
Mivel a monitorunk 2 dimenziós, a lakások 8 különböző tulajdonságát nem tudjuk egyszerre közvetlenül felrajzolni. 
A **Főkomponens-elemzés (PCA)** a matematikai forgatás révén kiválasztja a két legfontosabb információs tengelyt (a variancia ~60-70%-át), és levetíti a pontokat egy 2D síkra:
- A pontok színe a gépi tanulás által kiosztott klasztert jelöli.
- Figyelje meg, milyen szépen, felhőszerűen elkülönülnek a színes csoportok egymástól! Nincs kaotikus átfedés, ami igazolja az algoritmus precíz munkáját.

**💡 Fő tanulság**: A mesterséges intelligencia emberi beavatkozás nélkül is tökéletesen azonosította a valós piaci szegmenseket.

In [4]:
# Összefoglaló statisztika táblázat
cluster_summary = df_km.groupby('klaszter_nev')[cluster_vars].mean().reset_index()
display(HTML("<b>Klaszterek átlagos jellemzői:</b><br>" + cluster_summary.round(1).to_html(classes='table table-bordered table-striped', index=False)))

# Standardizált radar ábra a klaszterprofilokhoz
fig_radar = go.Figure()
scaler_radar = StandardScaler()
df_radar_scaled = pd.DataFrame(scaler_radar.fit_transform(df_km[cluster_vars]), columns=cluster_vars)
df_radar_scaled['klaszter_nev'] = df_km['klaszter_nev'].values
radar_agg = df_radar_scaled.groupby('klaszter_nev')[cluster_vars].mean().reset_index()

for i, row in radar_agg.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=row[cluster_vars].values,
        theta=cluster_vars,
        fill='toself',
        name=row['klaszter_nev']
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    showlegend=True,
    title='Standardizált Klaszterprofilok (Radar Diagram)',
    template=PLOTLY_TEMPLATE,
    height=500
)
fig_radar.show()

# Kereszttábla városrészek szerint
ct = pd.crosstab(df_km['varosresz'], df_km['klaszter_nev'])

fig3 = px.imshow(
    ct,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Városrészek és ingatlanpiaci klaszterek kereszttáblája (darabszám)',
    labels=dict(x="Ingatlan Szegmens", y="Városrész", color="Darabszám"),
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450, xaxis_tickangle=-30)
fig3.show()

klaszter_nev,nm_ar_huf,alapterulet_nm,szobaszam_osszes,allapot_kod,epulet_kora_ev
1. Új építésű prémium kis lakások (átlagkor: ~5 év),1555950.2,50.6,2.2,5.6,5.5
"2. Régebbi kompakt lakások (Panel/Tégla átlag, ~60 év)",1187207.5,43.5,1.7,2.8,59.7
"3. Idősebb nagyméretű lakások (Kedvező fajlagos ár, ~55 év)",1045639.1,73.5,3.1,2.9,54.7
4. Újszerű nagyméretű családi prémium (átlagkor: ~8 év),1408677.9,97.6,4.0,5.1,8.3


### 4. Városrész és Klaszter Kereszttábla Hőtérkép

**📌 Mit mutat a hőtérképes kereszttábla?**
A sorok a kőbányai városrészeket, az oszlopok a 4 azonosított lakástípust képviselik. A színek intenzitása azt mutatja meg, hol koncentrálódnak az adott típusú lakások:
- **Újhegy sora szinte világít a 2. klaszternél (családi panel)**: Itt szinte semmi más nincs, a kínálat homogén.
- **Óhegy sora a 3. és 4. klaszternél sűrűsödik (tégla és prémium zöldövezet)**: A minőségi szegmens fellegvára.
- **Városközpont és Ligettelek vegyes képet mutat**: A belépő kislakások és a felújítandó nagypolgári otthonok vegyülnek.

**💡 Befektetői iránytű**: Ha panelt keres, azonnal Újhegyre vagy a Városközpontba érdemes mennie; ha zöldövezeti téglát, Óhegy a kizárólagos célpont.

In [5]:
w_k_slider = widgets.IntSlider(min=2, max=6, value=4, description='Klaszter K:')
out_km = widgets.Output()

def frissit_km(*args):
    k = w_k_slider.value
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    s_score = silhouette_score(X_scaled, labels)
    
    with out_km:
        clear_output(wait=True)
        display(HTML(kpi_grid_html([
            ("Klaszterszám", f"{k} db", "Választott K", "#2563eb"),
            ("Silhouette Index", f"{s_score:.3f}", "Elkülönültség", "#10b981")
        ])))
        fig = px.scatter(
            x=coords_pca[:, 0], y=coords_pca[:, 1], color=[f'Klaszter {c+1}' for c in labels],
            title=f'K={k} Klaszter PCA vetülete (Silhouette={s_score:.3f})',
            template=PLOTLY_TEMPLATE
        )
        fig.show()

w_k_slider.observe(frissit_km, names='value')
display(w_k_slider)
display(out_km)
frissit_km()

IntSlider(value=4, description='Klaszter K:', max=6, min=2)

Output()